In [ ]:
import os
import cv2
import numpy as np

def load_dataset(data_dir):
    images = []
    labels = []

    for label in os.listdir(data_dir):
        class_path = os.path.join(data_dir, label)
        for img in os.listdir(class_path):
            img_path = os.path.join(class_path, img)
            image = cv2.imread(img_path)
            if image is not None:
                images.append(image)
                labels.append(int(label))

    return images, np.array(labels)

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 224

# Green Channel
def extract_green_channel(image):
    return image[:, :, 1]

# Resize + Crop + Normalize
def preprocess_image(image):
    # Resize
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    # Center Crop
    h, w = image.shape[:2]
    crop_size = min(h, w)
    start_x = w//2 - crop_size//2
    start_y = h//2 - crop_size//2
    image = image[start_y:start_y+crop_size, start_x:start_x+crop_size]

    # Green channel
    image = extract_green_channel(image)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    image = clahe.apply(image)

    # Noise removal
    image = cv2.GaussianBlur(image, (3,3), 0)

    # Normalize
    image = image / 255.0

    # Expand dims
    image = np.expand_dims(image, axis=-1)

    return image

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def get_augmentation():
    return ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        brightness_range=[0.8,1.2]
    )